Run version 1.0 - 01-12-2025

In [2]:
import subprocess
import json
import pandas as pd
from langdetect import detect

In [3]:
allowed_langs = ["en", "nl"]

start_date = "2019-08-01"
end_date = "now"

# which genres to KEEP, as given in categories.js
genres_list = ['APPLICATION', 'BEAUTY', 'BOOKS_AND_REFERENCE', 'COMMUNICATION', 'BUSINESS', 'EDUCATION', 'FOOD_AND_DRINK', 'HEALTH_AND_FITNESS', 'LIFESTYLE', 'MEDICAL', 
               'PARENTING', 'FAMILY']

In [4]:
with open("keywords.txt", encoding="utf-8") as f:
    keywords = [word.strip() for word in f.read().split(",") if word.strip()]

# for testing
keywords2 = ["dental", "tooth"]


In [6]:
# empty list to append results of different keywords to
all_results = []

for term in keywords:

    # load data as json using js for api
    result = subprocess.run(
        ["node", "nodejs/playstore.js", term],
        capture_output=True,
        text=True,
        check=True,
        encoding= "utf-8",
        errors="replace"
    )

    data = json.loads(result.stdout)

    result_count = len(data)

    # check if there are more or less than 250 results per keyword
    print(f"Query '{term}' returned {result_count} results")
    if result_count >= 250:
        print(f"!WARNING! for {term}: hit the 250 API limit")
    
    # add the results for which searchterm
    for result_term in data:
        result_term["query_term"] = term
    
    # append to the empty list
    all_results.extend(data)

df = pd.DataFrame(all_results)
print(f"Total of {df.shape[0]} apps found in initial search for {len(keywords)} keywords")

Query 'dental' returned 247 results
Query 'tooth' returned 245 results
Query 'teeth' returned 244 results
Query 'mouth' returned 249 results
Query 'oral' returned 247 results
Query 'gums' returned 240 results
Query 'dental health' returned 245 results
Query 'dental hygiene' returned 245 results
Query 'dental self-care' returned 249 results
Query 'dental cleaning' returned 244 results
Query 'tooth health' returned 248 results
Query 'tooth hygiene' returned 245 results
Query 'tooth self-care' returned 249 results
Query 'tooth cleaning' returned 228 results
Query 'teeth health' returned 246 results
Query 'teeth hygiene' returned 247 results
Query 'teeth self-care' returned 250 results
!WARNING! for teeth self-care: hit the 250 API limit
Query 'teeth cleaning' returned 227 results
Query 'mouth health' returned 250 results
!WARNING! for mouth health: hit the 250 API limit
Query 'mouth hygiene' returned 247 results
Query 'mouth self-care' returned 248 results
Query 'mouth cleaning' returned 

In [7]:
initial_amount = df.shape[0]
print(f"Initial amount in dataset: {initial_amount}")

# remove duplicates
df = df.drop_duplicates(subset="appId", keep="first")
print(f"Filtered {initial_amount - df.shape[0]} duplicates")

# languague filter based on description language
def detect_language_safe(text):
    try:
        if not isinstance(text, str) or text.strip() == "":
            return "unknown"
        return detect(text)
    except Exception:
        return "unknown"

# Apply detection to DataFrame
df["detected_lang"] = df["description"].apply(detect_language_safe)

# Filter based on allowed languages
filter_lang = df["detected_lang"].apply(lambda x: x in allowed_langs or x == "unknown")

num_filtered_out = (~filter_lang).sum()
print(f"Filtered {num_filtered_out} apps with description not in {allowed_langs} language")

df = df[filter_lang]

# filter title or description includes any of the keywords
# create regex pattern
pattern = "|".join(keywords)
filtered_keywords = (
    df["title"].str.lower().str.contains(pattern, regex=True, na=False) |
    df["description"].str.lower().str.contains(pattern, regex=True, na=False)
)
print(f"Filtered {(~filtered_keywords).sum()} apps that do NOT contain any of the given keywords")
df = df[filtered_keywords]


# filter on genre, see categories.js for a full list of all categories available in the play store
# Number of rows before filtering
before_count = len(df)

#filtering
df = df[df['genreId'].isin(genres_list)]

# Number of rows after filtering
after_count = len(df)

# How many rows were removed
filtered_out_cat = before_count - after_count
print(f"Filtered {filtered_out_cat} apps with categories other than {genres_list}")

# filter based on release date of latest version
start_len = len(df)

def parse_bound(s: str) -> pd.Timestamp:
    if s.lower() == "now":
        # Use today's calendar date (local) as the bound
        return pd.Timestamp.today().normalize()
    # Strict YYYY-MM-DD as requested
    return pd.to_datetime(s, format="%Y-%m-%d")

# Parse bounds and make end inclusive by pushing exclusive bound to next midnight
start = parse_bound(start_date)
end = parse_bound(end_date)
end_exclusive = end + pd.Timedelta(days=1)

# Convert updated (ms since epoch). Keep tz-naive for simple comparison.
updated_dt = pd.to_datetime(df["updated"], unit="ms", errors="coerce", utc=True).dt.tz_convert(None)

# Convert released
released_dt = pd.to_datetime(df["released"], format="%b %d, %Y", errors="coerce")

# Effective date: prefer updated, else released
effective_dt = updated_dt.fillna(released_dt)

# Keep rows IN the interval [start, end] (inclusive)
in_range = (effective_dt >= start) & (effective_dt < end_exclusive)

#update df
df = df[in_range]

#change format of updated column
df["updated"] = updated_dt[in_range].dt.strftime("%b %d, %Y")

filtered_out_date = start_len - len(df)
print(f"Filtered {filtered_out_date} apps that were not in the specifie date range")

print(f"Results after filtering: {df.shape[0]}")


Initial amount in dataset: 13437
Filtered 7834 duplicates
Filtered 54 apps with description not in ['en', 'nl'] language
Filtered 2903 apps that do NOT contain any of the given keywords
Filtered 1045 apps with categories other than ['APPLICATION', 'BEAUTY', 'BOOKS_AND_REFERENCE', 'COMMUNICATION', 'BUSINESS', 'EDUCATION', 'FOOD_AND_DRINK', 'HEALTH_AND_FITNESS', 'LIFESTYLE', 'MEDICAL', 'PARENTING', 'FAMILY']
Filtered 416 apps that were not in the specifie date range
Results after filtering: 1185


In [8]:
df.to_excel("android_data_v1_0.xlsx", index=False)